# Recommender System In-Depth Analysis
This notebook analyzes:
1. **Cold-Start Performance**: How models perform for new/low-activity users vs active users.
2. **Popularity Bias & Diversity**: How frequently models recommend blockbusters vs discovering long-tail movies.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data import load_ratings, temporal_train_test_split, get_user_relevant_test_items
from src.metrics import rmse, precision_at_k, recall_at_k, ndcg_at_k
from src.models.popularity import AverageRatingBaseline, PopularityBaseline
from src.models.knn import ItemKNNRecommender
from src.models.mf import PyTorchMatrixFactorization

## 1. Load Data & Split

In [ ]:
df = load_ratings()
train_df, test_df = temporal_train_test_split(df, test_ratio=0.20)
test_relevants = get_user_relevant_test_items(test_df, min_rating=3.5)

# User activity in training
user_train_counts = train_df.groupby('userId')['rating'].count()

## 2. Fit Models

In [ ]:
pop_model = PopularityBaseline().fit(train_df)
knn_model = ItemKNNRecommender(n_neighbors=20).fit(train_df)
mf_model = PyTorchMatrixFactorization(embedding_dim=32, epochs=8).fit(train_df)

## 3. Analysis A: Cold-Start vs Heavy Users
Segment users by number of train ratings: Low (<= 20), Medium (21-100), High (> 100).

In [ ]:
def eval_user_group(model, user_subset, test_relevants, k=10):
    ndcgs = []
    recs = model.recommend_for_users(user_subset, k=k, exclude_seen=True)
    for u in user_subset:
        if u in test_relevants and len(test_relevants[u]) > 0:
            ndcgs.append(ndcg_at_k(recs[u], test_relevants[u], k=k))
    return np.mean(ndcgs) if ndcgs else 0.0

low_users = user_train_counts[user_train_counts <= 20].index.tolist()
med_users = user_train_counts[(user_train_counts > 20) & (user_train_counts <= 100)].index.tolist()
high_users = user_train_counts[user_train_counts > 100].index.tolist()

groups = {'Low (<=20)': low_users, 'Med (21-100)': med_users, 'High (>100)': high_users}
summary = []
for g_name, u_list in groups.items():
    summary.append({
        'Group': g_name,
        'Users': len(u_list),
        'Popularity NDCG@10': eval_user_group(pop_model, u_list, test_relevants),
        'Item-KNN NDCG@10': eval_user_group(knn_model, u_list, test_relevants),
        'PyTorch MF NDCG@10': eval_user_group(mf_model, u_list, test_relevants),
    })

cold_start_df = pd.DataFrame(summary)
cold_start_df

## 4. Analysis B: Popularity Bias & Novelty
We calculate the average training rating count of recommended movies. High average count = strong popularity bias.

In [ ]:
item_popularity = train_df.groupby('movieId')['rating'].count().to_dict()
sample_users = list(test_relevants.keys())[:200]

def average_recommendation_popularity(model, users, k=10):
    recs = model.recommend_for_users(users, k=k, exclude_seen=True)
    all_pop = []
    for u in users:
        for movie in recs[u]:
            all_pop.append(item_popularity.get(movie, 0))
    return np.mean(all_pop)

print('Avg popularity score of recommended items (higher = more popular bias):')
print(f'Popularity baseline: {average_recommendation_popularity(pop_model, sample_users):.1f}')
print(f'Item-KNN:           {average_recommendation_popularity(knn_model, sample_users):.1f}')
print(f'PyTorch MF:         {average_recommendation_popularity(mf_model, sample_users):.1f}')